In [1]:
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor,LogitsProcessorList

import torch
from transformers import BitsAndBytesConfig
from tqdm import tqdm
from pathlib import Path
import datasets

/work/pi_mccallum_umass_edu/aparashar_umass_edu/.conda/envs/as01/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
qa_exemplars = [
  {
    "question": "What do people use to absorb extra ink from a fountain pen? Answer Choices: (a) shirt pocket (b) calligrapher’s hand (c) inkwell (d) desk drawer (e) blotter",
    "reasoning": "The answer must be an item that can absorb ink. Of the above choices, only blotters are used to absorb ink. So the answer is (e)",
    "answer": "e"
  },
  {
    "question": "What home entertainment equipment requires cable? Answer Choices: (a) radio shack (b) substation (c) television (d) cabinet",
    "reasoning": "The answer must require cable. Of the above choices, only television requires cable. So the answer is (c)",
    "answer": "c"
  },
  {
    "question": "The fox walked from the city into the forest, what was it looking for? Answer Choices: (a) pretty flowers (b) hen house (c) natural habitat (d) storybook",
    "reasoning": "The answer must be something in the forest. Of the above choices, only natural habitat is in the forest. So the answer is (c).",
    "answer": "c"
  },
  {
    "question": "Sammy wanted to go to where the people were. Where might he go? Answer Choices: (a) populated areas (b) race track (c) desert (d) apartment (e) roadblock",
    "reasoning": "The answer must be a place with a lot of people. Of the above choices, only populated areas have a lot of people. So the answer is (a).",
    "answer": "a"
  },
  {
    "question": "Where do you put your grapes just before checking out? Answer Choices: (a) mouth (b) grocery cart (c) super market (d) fruit basket (e) fruit market",
    "reasoning": "The answer should be the place where grocery items are placed before checking out. Of the above choices, grocery cart makes the most sense for holding grocery items. So the answer is (b)",
    "answer": "b"
  },
  {
    "question": "Google Maps and other highway and street GPS services have replaced what? Answer Choices: (a) united states (b) mexico (c) countryside (d) atlas",
    "reasoning": "The answer must be something that used to do what Google Maps and GPS services do, which is to give directions. Of the above choices, only atlases are used to give directions. So the answer is (d)",
    "answer": "d"
  },
  {
    "question": "Before getting a divorce, what did the wife feel who was doing all the work? Answer Choices: (a) harder (b) anguish (c) bitterness (d) tears (e) sadness",
    "reasoning": "The answer should be the feeling of someone getting divorced who was doing all the work. Of the above choices, the closest feeling is bitterness. So the answer is (c).",
    "answer": "c"
  }
]


In [3]:
from typing import Union,List

class TruncateLogitsProcessor(LogitsProcessor):
    def __init__(self,token_id: Union[int, List[int]],stop_word:Union[str, List[str]],eos_token_id: Union[int, List[int]],tokenizer):
        if isinstance(token_id, int):
            token_id = [token_id]
        if isinstance(eos_token_id, int):
            eos_token_id = [eos_token_id]
        # if not all(isinstance(i,int) for i in token_id) or any(i < 0 for i in token_id):
        #     logger.warning(f"`token_id` has to be a list of positive integers, but is {token_id}")
        self.stop_word = stop_word
        self.token_id = token_id
        
        self.eos_token_id = eos_token_id
        self.tokenizer = tokenizer
    def __call__(self,input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        cur_len = input_ids.shape[-1]
        
        # print(cur_len)
        # print(scores.shape)
        max_score_ids = torch.argmax(scores, dim=1)
        
        for i in range(len(max_score_ids)):
            #print(tokenizer.decode(max_score_ids[i]))
            if (max_score_ids[i] in tokenizer(self.stop_word).input_ids or self.tokenizer.decode(max_score_ids[i]).find(self.stop_word) != -1) :
                print("GITTT")
                scores[i][:] = -float('inf')
                scores[i][self.eos_token_id[0]] = 0
        #print(torch.argmax(scores, dim=1))
        # print(input_ids.shape)
        #print(input_ids)
        scores.to(input_ids.device)
        if torch.argmax(scores[:,]) in [894]:
            print('yes')
            scores = torch.zeros(scores.shape)
            scores[:, self.eos_token_id] = 1
                
        return scores



In [15]:
data[0]

{'id': '1afa02df02c908a558b4036e80242fac',
 'question': 'A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?',
 'question_concept': 'revolving door',
 'choices': {'label': ['A', 'B', 'C', 'D', 'E'],
  'text': ['bank', 'library', 'department store', 'mall', 'new york']},
 'answerKey': 'A'}

In [11]:
data[0]
d = data[0]
    
    # prompt_arr = [default_fwd_input_prefix] +[default_fwd_question_prefix + d['input']]+[default_fwd_reasoning_prefix]
options_text = ['('+ l.lower() + ') ' + t  for l,t in zip(d['choices']['label'],d['choices']['text'])]
' '.join(options_text)

'(a) bank (b) library (c) department store (d) mall (e) new york'

In [13]:
# default_instruction = "Answer the question with Yes or No: "
default_fwd_input_prefix = """Like each of the previous examples, answer the question with either of the answer choices provided.
Provide reasoning for your answer.
End your reasoning with the sentence: "So the answer is <your_answer_choice>". Replace "<your_answer_choice>" with your final answer, which should be one of the answer choice label letters (a,b,c,d,e,etc.)."""
turn_prefix_model = '<start_of_turn>model'
turn_prefix_human = '<start_of_turn>human'
turn_suffix = '<end_of_turn>'
default_fwd_demos_prefix = "Here are some examples of reasoning based question answers."
default_fwd_reasoning_prefix = "Reasoning: "
default_fwd_question_prefix = "Question: "
default_fwd_target_prefix = "Answer: "
demos = random.sample(qa_exemplars,6)
demos = [[default_fwd_question_prefix + d["question"]]+ [default_fwd_reasoning_prefix + d["reasoning"] + default_fwd_target_prefix + d["answer"]] for d in demos]# def test(prompt_arr, model, tokenizer, data,demos, default_fwd_target_prefix, N = 10, temp = 0.5):
bnb_config= BitsAndBytesConfig(
            load_in_8bit=True,)
output_dict = {}
import json
filepath = "/work/pi_dhruveshpate_umass_edu/aparashar_umass_edu/Arithmetic-Sampling/data/stratqa_data.json"
datasets.config.DOWNLOADED_DATASETS_PATH = Path('/work/pi_dhruveshpate_umass_edu/aparashar_umass_edu/datasets')
dataset_name = 'tau/commonsense_qa'
split = 'validation'
data = datasets.load_dataset(dataset_name,'', split=split,cache_dir ='/work/pi_dhruveshpate_umass_edu/aparashar_umass_edu/datasets/.cache' )
        

demo_questions = [d['question'] for d in qa_exemplars]
print(len(data))
# data = [i for i in data if i['input'] not in demo_questions]
print(len(data))
# device_map="auto"
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b", use_fast=True)
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b",
                                             # torch_dtype=torch.bfloat16,
                                             cache_dir = '/work/pi_mccallum_umass_edu/aparashar_umass_edu/models/.cache',
                                             # device_map="auto",
                                                # quantization_config=bnb_config
                                            ).to('cuda')
logits_processor = LogitsProcessorList(
    [
        #StopAfterTokenLogitsProcessor(token_id_to_stop=29973,max_length=40),
        TruncateLogitsProcessor(stop_word='Question',token_id=[tokenizer.encode('Question')],eos_token_id=tokenizer.eos_token_id,tokenizer=tokenizer),
    ])
# torch.device = 'cuda'
# set pad_token_id to eos_token_id because GPT2 does not have a EOS token
model.config.pad_token_id = model.config.eos_token_id
model.generation_config.pad_token_id = model.config.eos_token_id
for  idx,d in enumerate(tqdm(data, desc="Predicting")):
    print(d)
    d = data[0]
    options = ['('+ l.lower() + ') ' + t  for l,t in zip(d['choices']['label'],d['choices']['text'])]
    options_text = 'Answer choices: ' +' '.join(options_text)
    d['question'] += options_text
    # prompt_arr = [default_fwd_input_prefix] +[default_fwd_question_prefix + d['input']]+[default_fwd_reasoning_prefix]
    # options_text = [['('+ l[i] + ') ' + t[i] for i in len(d['choices']['labels'])] for l,t in zip(d['choices']['label'],d['choices']['text'])]
    prompt_arr = [default_fwd_input_prefix] +[default_fwd_question_prefix + d['question']]
    prompt_arr = demos + [prompt_arr]
    prompt_arr = [element for sublist in prompt_arr for element in sublist]
    # print(prompt_arr)
    input_prompt = turn_prefix_human + (' \n').join(prompt_arr)
    print(f"prompt: {input_prompt}")
    input_ids = tokenizer(input_prompt, truncation=True, return_tensors="pt").input_ids.to('cuda')
    outputs_arith = model.generate(
        input_ids = input_ids,
        num_return_sequences = 5,
        do_sample = True,
        max_new_tokens = 200,
        temperature = 1.,
        num_beams = 1,
        return_dict_in_generate=True,
        logits_processor=logits_processor,
        use_arithmetic = True
        )
    outputs_sample = model.generate(
        input_ids = input_ids,
        num_return_sequences = 5,
        do_sample = True,
        temperature = 1.,
        num_beams = 1,
        max_new_tokens = 200,
        return_dict_in_generate=True,
        logits_processor=logits_processor,
        
        use_arithmetic = False
        )
    print(input_ids.shape[1])
    outputs_arith = [tokenizer.decode(o, skip_special_tokens=True)[len(input_prompt):] for o in outputs_arith.sequences]
    outputs_sample = [tokenizer.decode(o, skip_special_tokens=True)[len(input_prompt):] for o in outputs_sample.sequences]
    # print(f"Arithmetic output: {tokenizer.batch_decode(outputs_arith[input_ids.shape[1]:], skip_special_tokens=True)}")
    outputs_arith_str = '\n###\n'.join(outputs_arith)
    print("---")
    outputs_sample_str = '\n###\n'.join(outputs_sample)
    print(f"Arithmetic output: {outputs_arith_str}")
    print(f"Sample output: {outputs_sample_str}")
    

1221
1221


Predicting:   0%|                                                                                                                                                             | 0/1221 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'id': '1afa02df02c908a558b4036e80242fac', 'question': 'A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?', 'question_concept': 'revolving door', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['bank', 'library', 'department store', 'mall', 'new york']}, 'answerKey': 'A'}
prompt: <start_of_turn>humanQuestion: The fox walked from the city into the forest, what was it looking for? Answer Choices: (a) pretty flowers (b) hen house (c) natural habitat (d) storybook 
Reasoning: The answer must be something in the forest. Of the above choices, only natural habitat is in the forest. So the answer is (c).Answer: c 
Question: Google Maps and other highway and street GPS services have replaced what? Answer Choices: (a) united states (b) mexico (c) countryside (d) atlas 
Reasoning: The answer must be something that used to do what Google Maps and GPS services do, which is to give directions. Of the above choices, only atlases are 

Predicting:   0%|                                                                                                                                                   | 1/1221 [00:07<2:39:02,  7.82s/it]

GITTT
634
---
Arithmetic output:  city (f) supermarket 
Reasoning: The revolving door helps with direction, but it also serves as a security measure at banks, libraries, department stores, shopping malls, and supermarkets. Of the above choices, only department stores are also used to be both convenient and serve as a form of security measure. So the answer choice is (c).Answer: c 

###
 cityAnswer: d
###
 city 
Reasoning: The answer for this one must be a place which serves as a security measure at a point where a lot of people would travel. Of the above choices, only a mall serves as a safety measure at a place where a lot of travelers need security. So the answer is (e).Answer: e

###
 city 
Reasoning: The answer must be a building with a lot of people inside that needs to make a lot of turns. Of the above choices, only new york city serves as a place where a lot of people are and makes a lot of turns. So the answer is (e).Answer: e 

###
 city station 
Reasoning: The answer should b

...

2185
---
Arithmetic output:                                                                                                                                                                                                                                                                t                                                                                                                                                                                                                                                               b                                                                                                                                                                                                                                                                               i                                                                                                                                                                                            

Predicting:   1%|█▏                                                                                                                                                | 10/1221 [02:58<8:29:42, 25.25s/it]

3689
---
Arithmetic output:                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         <end of game>
                                                                 </game_intro_section>

###
                                                                                                                                                                                                                                                                                                                                                                                          

Predicting:   1%|█▏                                                                                                                                                | 10/1221 [03:10<6:24:48, 19.07s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 31.94 GiB. GPU 0 has a total capacty of 44.35 GiB of which 29.34 GiB is free. Including non-PyTorch memory, this process has 15.00 GiB memory in use. Of the allocated memory 10.75 GiB is allocated by PyTorch, and 3.94 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [10]:
data[0]

{'id': '1afa02df02c908a558b4036e80242fac',
 'question': 'A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?',
 'question_concept': 'revolving door',
 'choices': {'label': ['A', 'B', 'C', 'D', 'E'],
  'text': ['bank', 'library', 'department store', 'mall', 'new york']},
 'answerKey': 'A'}